# Running Code Before and After Model Calls in LangChain Agents

### Introduction

In this notebook, we will learn how to run code before and after model calls in LangChain agents using `@before_model` and `@after_model` middleware. This allows us to access and manipulate the agent's short-term memory (state) at different stages of the agent's execution — for example to trim the message history before a call, or to validate and filter the model's output after a call.

![](./images/before_after_model.png)

### Prerequisites

- Completion of steps in `010_langchain_basics.ipynb` where we deployed an Azure Foundry with an LLM model.
- A self-hosted LLM (vLLM on Azure Container Apps) and/or an Azure Foundry deployment (see the `infra` folder).

### Getting the LLM Endpoint and API Key

To use the LLM model with `langchain`, we first need to retrieve the self-hosted LLM endpoint, the Foundry endpoint and API key from the Terraform outputs. You can do this by running the following command.


In [ ]:
foundry_endpoint = ! terraform output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name = ! terraform output -raw llm_model_deployment_name
llm_model_deployment_name = llm_model_deployment_name.n
print("LLM Model Deployment Name :", llm_model_deployment_name)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4


### 1. Set Up the LLM Model

Then we can test the endpoint and key by making a simple API call to the LLM model.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name,
    streaming=True,
    max_completion_tokens=512
)

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

### 2. Before Model

Access short-term memory (state) in `@before_model` middleware to process messages before model calls.

In [ ]:
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langchain_core.runnables import RunnableConfig
from langgraph.runtime import Runtime
from typing import Any


@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Keep only the last few messages to fit context window."""
    messages = state["messages"]

    if len(messages) <= 3:
        return None  # No changes needed

    first_msg = messages[0]
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }


agent = create_agent(
    model=model,
    tools=[],
    middleware=[trim_messages],
    checkpointer=InMemorySaver()
)

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

agent.invoke({"messages": "hi, my name is bob"}, config)
agent.invoke({"messages": "write a short poem about cats"}, config)
agent.invoke({"messages": "now do the same but for dogs"}, config)

response = agent.invoke({"messages": "what's my name?"}, config)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

hi, my name is bob
================================== Ai Message ==================================

Soft paws whisper through the night,  
Moonlit eyes shine calm and bright.  
Curled in warmth, they dream and play,  
Tiny kings of every day.
================================ Human Message =================================

now do the same but for dogs
================================== Ai Message ==================================

Bright eyes shining, tail in flight,  
Joyful paws race through the light.  
Faithful hearts in games they stay,  
Happy friends of every day.
================================ Human Message =================================

what's my name?
================================== Ai Message ==================================

Your name is Bob.


### 3. After Model

Access short-term memory (state) in `@after_model` middleware to process messages after model calls.

In [ ]:
from langchain.messages import RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import after_model
from langgraph.runtime import Runtime


@after_model
def validate_response(state: AgentState, runtime: Runtime) -> dict | None:
    """Remove messages containing sensitive words."""
    STOP_WORDS = ["password", "secret"]
    last_message = state["messages"][-1]

    if any(word in last_message.content for word in STOP_WORDS):
        return {"messages": [RemoveMessage(id=last_message.id)]}
    
    return None

agent = create_agent(
    model=model,
    tools=[],
    middleware=[validate_response],
    checkpointer=InMemorySaver(),
)

response = agent.ainvoke({"messages": "write a password in the response"})


### More Resources

- [LangChain Middleware](https://docs.langchain.com/oss/python/langchain/middleware) — overview of agent middleware
- [Before/After Model Hooks](https://docs.langchain.com/oss/python/langchain/middleware#before_model) — the hooks used in this notebook
- [LangGraph Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) — checkpointers and state
- [LangGraph ReAct Agent](https://python.langchain.com/docs/tutorials/agents/)
- [ChatOpenAI with custom endpoints](https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html)
